## Linearize FCDF triples for KG-RAG

This notebook reads `Bundesliga23_24_test.ttl`, converts each RDF triple into a simple text line, and writes the output to JSONL for downstream KG-RAG indexing.

In [1]:
# Install dependency (run once)
!pip install -q rdflib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import json

from rdflib import Graph, URIRef, Literal
from rdflib.namespace import RDF, RDFS

project_root = Path(".").resolve()
ttl_path = project_root / "Bundesliga23_24_test.ttl"
output_path = project_root / "Bundesliga23_24_test_triples_linearized.jsonl"

In [3]:
project_root

WindowsPath('C:/Users/dyury/Desktop/Master Thesis')

In [ ]:
def local_name(uri: str) -> str:
    """Return URI tail after # or last /."""
    if "#" in uri:
        return uri.rsplit("#", 1)[-1]
    return uri.rstrip("/").rsplit("/", 1)[-1]

def get_label(node, g: Graph) -> str:
    """Use rdfs:label when available, otherwise fallback to URI tail/literal text."""
    if isinstance(node, Literal):
        return str(node)
    if isinstance(node, URIRef):
        lbl = next(g.objects(node, RDFS.label), None)
        if lbl is not None:
            return str(lbl)
        return local_name(str(node))
    return str(node)

def linearize_triple(s, p, o, g: Graph) -> str:
    """Format one triple as: subject predicate object."""
    s_label = get_label(s, g)
    p_label = get_label(p, g)
    o_label = get_label(o, g)
    return f"{s_label} {p_label} {o_label}"

In [5]:
# Parse TTL and write linearized triples to JSONL
g = Graph()
g.parse(ttl_path, format="turtle")
print("Triples in graph:", len(g))

count = 0
with open(output_path, "w", encoding="utf-8") as f:
    for s, p, o in g:
        text = linearize_triple(s, p, o, g)
        record = {
            "text": text,
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        count += 1

Triples in graph: 46725


In [6]:
# Preview a few linearized triples
preview_n = 5
with open(output_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= preview_n:
            break
        rec = json.loads(line)
        print(rec["text"])

event/62ead0ab-e3ec-46fa-ab33-7b9d0473a344 x 48.6
match/3895052 events event/ce261d4b-9c4d-4414-a519-e95dea8a0fa3
event/7d2cc416-583e-4918-a35e-429b12a4aa83 y_end 70.0
event/d7f33026-3427-45b8-b838-3801ffb99a25 receiver_time 01:19:16.477
event/a8aa1263-3b1e-4640-89e7-434c09d27f53 event_period second_half
